In [ ]:
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install sentencepiece
!pip install pandas
!pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.neural_network import MLPClassifier

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv("/content/hindi_fake_news.csv")

data = data[['text','label']]
data = data.dropna()

data.head()

,text,label
0,‘मोदी के शासन के दौरान गंगा’ गंगा नदी नरेन्द्...,1
1,यह खबर आने से पहले छवि क्रेडिट जस्टिन सुलिवान/...,1
2,गुलाब गेंद वाल डे-नाइट टेस्ट मैच कप्ता विराट क...,0
3,उत्तर कोरिया रॉकेट प्रक्षेपण योजनाएं 71 0 15 0...,1
4,राष्ट्रपति डोनाल्ड ट्रम्प और प्रथम महिला मेलान...,0


In [74]:
print(data.head(2))

                                                text  label
0  ‘मोदी के शासन के दौरान गंगा’  गंगा नदी नरेन्द्...      1
1  यह खबर आने से पहले छवि क्रेडिट जस्टिन सुलिवान/...      1


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data["text"],
    data["label"],
    test_size=0.2,
    random_state=42
)

MODEL 1 — NAIVE BAYES

In [ ]:
vectorizer = CountVectorizer(max_features=4000)

X_train_nb = vectorizer.fit_transform(train_texts)
X_test_nb = vectorizer.transform(test_texts)

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train_nb, train_labels)

MultinomialNB()

In [ ]:
nb_pred = nb_model.predict(X_test_nb)

nb_acc = accuracy_score(test_labels, nb_pred)

print("Naive Bayes Accuracy:", nb_acc)
print(classification_report(test_labels, nb_pred))

Naive Bayes Accuracy: 0.7766423357664234
              precision    recall  f1-score   support

           0       0.77      0.88      0.82      1985
           1       0.79      0.64      0.71      1440

    accuracy                           0.78      3425
   macro avg       0.78      0.76      0.76      3425
weighted avg       0.78      0.78      0.77      3425



MODEL 2 — MLP (Neural Network)

In [ ]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128,64,32),
    activation='relu',
    solver='adam',
    max_iter=20
)

mlp_model.fit(X_train_nb, train_labels)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=20)

In [ ]:
mlp_pred = mlp_model.predict(X_test_nb)

mlp_acc = accuracy_score(test_labels, mlp_pred)

print("MLP Accuracy:", mlp_acc)
print(classification_report(test_labels, mlp_pred))

MLP Accuracy: 0.8017518248175183
              precision    recall  f1-score   support

           0       0.81      0.85      0.83      1985
           1       0.78      0.73      0.76      1440

    accuracy                           0.80      3425
   macro avg       0.80      0.79      0.79      3425
weighted avg       0.80      0.80      0.80      3425



*MODEL* 3 — XLM-RoBERTa (SOTA)

In [ ]:
xlm_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def xlm_tokenize(example):

    return xlm_tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset_xlm = Dataset.from_dict({
    "text": list(train_texts),
    "label": list(train_labels)
})

test_dataset_xlm = Dataset.from_dict({
    "text": list(test_texts),
    "label": list(test_labels)
})

train_dataset_xlm = train_dataset_xlm.map(xlm_tokenize, batched=True)
test_dataset_xlm = test_dataset_xlm.map(xlm_tokenize, batched=True)

Map:   0%|          | 0/13698 [00:00<?, ? examples/s]

Map:   0%|          | 0/3425 [00:00<?, ? examples/s]

In [ ]:
xlm_model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='binary'
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/results_xlm",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    save_steps=1000000
)

In [ ]:
trainer_xlm = Trainer(
    model=xlm_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer_xlm.train()

Step,Training Loss
500,0.558526
1000,0.415571
1500,0.380557
2000,0.354356
2500,0.328849
3000,0.298196


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3428, training_loss=0.3772574439210124, metrics={'train_runtime': 1383.354, 'train_samples_per_second': 39.608, 'train_steps_per_second': 2.478, 'total_flos': 3604095236321280.0, 'train_loss': 0.3772574439210124, 'epoch': 4.0})

In [ ]:
trainer_xlm.save_model("/content/drive/MyDrive/xlm_roberta_fake_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
xlm_results = trainer_xlm.evaluate()

In [ ]:
print("XLM-RoBERTa Accuracy:", xlm_results["eval_accuracy"])

XLM-RoBERTa Accuracy: 0.8569343065693431


In [ ]:
print(xlm_results)

{'eval_loss': 0.4297473728656769, 'eval_accuracy': 0.8569343065693431, 'eval_precision': 0.8383190883190883, 'eval_recall': 0.8173611111111111, 'eval_f1': 0.8277074542897328, 'eval_runtime': 27.1805, 'eval_samples_per_second': 126.01, 'eval_steps_per_second': 7.91, 'epoch': 4.0}


**Testing**

In [62]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "/content/drive/MyDrive/xlm_roberta_fake_news_model"

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=2, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [64]:
text = "भारत सरकार ने सभी बैंक बंद करने की घोषणा कर दी है"

In [85]:
text = "भारत सरकार ने डिजिटल भुगतान को बढ़ावा देने के लिए नई पहल शुरू की है।"

In [86]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
)

In [87]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
pred = torch.argmax(logits, dim=1).item()

In [88]:
if pred == 0:
    print("Prediction: Fake News")
else:
    print("Prediction: Real News")

Prediction: Real News


In [89]:
probs = torch.softmax(logits, dim=1)

print("Fake probability:", probs[0][0].item())
print("Real probability:", probs[0][1].item())

Fake probability: 0.14696469902992249
Real probability: 0.8530353307723999
